# Spotify Support — DeBERTa-v3 Intent Classifier
**Final training script** — incorporates all stability fixes.

| | |
|---|---|
| Model | `microsoft/deberta-v3-small` (fallback: `roberta-base`) |
| Task | 8-class intent classification |
| Target | Val Accuracy ≥ 88% \| Macro F1 ≥ 0.85 |
| Runtime | ~20-25 min on free T4 GPU |
| Training | Native PyTorch loop (no Trainer) |

**Run cells top-to-bottom. Do NOT skip any cell.**

### Cell 1 — Install Dependencies (pinned versions)

In [ ]:
# Pin to last known-stable release for DeBERTa-v3 on T4
!pip install -q transformers==4.40.2 tokenizers==0.19.1 accelerate==0.29.3 scikit-learn

import torch
assert torch.cuda.is_available(), "No GPU! Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
import transformers
print(f"transformers: {transformers.__version__}")
print("All good — proceed to Cell 2.")

### Cell 2 — Upload `colab_bundle.zip`

In [ ]:
from google.colab import files
import zipfile, os

print("Upload colab_bundle.zip from your PC:")
uploaded = files.upload()

with zipfile.ZipFile('colab_bundle.zip', 'r') as z:
    z.extractall('.')

for f in ['intent_train.jsonl', 'intent_val.jsonl', 'golden_eval.jsonl']:
    size = os.path.getsize(f) / 1024
    print(f"  {f:<30} {size:.1f} KB")
print("Data ready.")

### Cell 3 — Config, Tokenizer & Dataset

In [ ]:
import os, json, time
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

# ── Configuration ──────────────────────────────────────────────────────────
MODEL_ID   = 'microsoft/deberta-v3-small'
DEVICE     = torch.device('cuda')
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-5
MAX_LEN    = 128
PATIENCE   = 3

INTENT_MAP = {
    'content_availability': 0,  'technical_support': 1,
    'account_access': 2,        'billing_and_payments': 3,
    'subscription_and_plans': 4,'platform_and_regional': 5,
    'product_feedback': 6,      'other_or_ambiguous': 7
}
ID_TO_LABEL = {v: k for k, v in INTENT_MAP.items()}
NUM_LABELS  = len(INTENT_MAP)

# ── Tokenizer ──────────────────────────────────────────────────────────────
print(f'--> Loading tokenizer from {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# ── Dataset class ──────────────────────────────────────────────────────────
class IntentDataset(Dataset):
    def __init__(self, path, tokenizer, max_len):
        self.records = [json.loads(l) for l in open(path, encoding='utf-8')]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        r   = self.records[idx]
        enc = self.tokenizer(
            r['text'], truncation=True, max_length=self.max_len,
            padding=False, return_tensors=None
        )
        return {
            'input_ids':      enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'label':          int(r['label'])
        }

# ── Collator: only input_ids + attention_mask + labels ─────────────────────
# token_type_ids intentionally excluded (causes NaN in deberta-v3 EMD layer)
def collate_fn(batch):
    max_len = max(len(b['input_ids']) for b in batch)
    pad_id  = tokenizer.pad_token_id

    input_ids, attention_masks, labels = [], [], []
    for b in batch:
        pad = max_len - len(b['input_ids'])
        input_ids.append(      b['input_ids']      + [pad_id] * pad)
        attention_masks.append(b['attention_mask'] + [0]      * pad)
        labels.append(b['label'])

    return {
        'input_ids':      torch.tensor(input_ids,       dtype=torch.long),
        'attention_mask': torch.tensor(attention_masks, dtype=torch.long),
        'labels':         torch.tensor(labels,          dtype=torch.long)
    }

train_ds = IntentDataset('intent_train.jsonl', tokenizer, MAX_LEN)
val_ds   = IntentDataset('intent_val.jsonl',   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32,         shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f'--> Train: {len(train_ds)} examples | Val: {len(val_ds)} examples')
print(f'    Steps per epoch: {len(train_loader)} | Total steps: {len(train_loader)*EPOCHS}')

### Cell 4 — Sanity Check (must pass before training)

In [ ]:
print('--> Sanity check: loading model and running one real batch...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, num_labels=NUM_LABELS,
    id2label=ID_TO_LABEL, label2id=INTENT_MAP
).to(DEVICE)

model.train()
batch = next(iter(train_loader))
batch = {k: v.to(DEVICE) for k, v in batch.items()}

with torch.no_grad():
    out = model(**batch)

loss_val = out.loss.item()
is_nan   = torch.isnan(out.loss).item()

print(f'  labels : {batch["labels"].tolist()}')
print(f'  logits : shape={out.logits.shape}, any_nan={torch.isnan(out.logits).any().item()}')
print(f'  loss   : {loss_val:.4f}')

if is_nan:
    print()
    print('FAIL: loss is NaN. Try switching model to roberta-base:')
    print('  → In Cell 3, change MODEL_ID = "roberta-base" and re-run from Cell 3')
    raise SystemExit('Sanity check failed — see instructions above.')
elif abs(loss_val - 2.07) > 0.5:
    print(f'WARNING: loss {loss_val:.4f} is unusual for random init (expected ~2.07). Proceed with caution.')
else:
    print(f'PASS: loss={loss_val:.4f} (expected ~2.07). Proceeding to training.')

### Cell 5 — Training (Native PyTorch Loop)

In [ ]:
# ── Optimizer + Scheduler ──────────────────────────────────────────────────
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(0.10 * total_steps)
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Optimizer : AdamW  lr={LR}  wd=0.01')
print(f'Scheduler : linear warmup {warmup_steps} steps / {total_steps} total')
print(f'Target    : accuracy >= 88%  |  macro_f1 >= 0.85')
print(f'EarlyStop : patience={PATIENCE} epochs')
print()

# ── Eval helper ────────────────────────────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch  = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**batch).logits
            all_preds.extend(torch.argmax(logits, dim=-1).cpu().tolist())
            all_labels.extend(batch['labels'].cpu().tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, f1, all_preds, all_labels

# ── Training Loop ──────────────────────────────────────────────────────────
best_f1, best_epoch, no_improve = 0.0, 0, 0
train_start = time.perf_counter()

print(f"{'Epoch':<7} {'Train Loss':<13} {'Val Acc %':<12} {'Macro F1':<12} {'Status'}")
print('─' * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, steps = 0.0, 0
    epoch_start = time.perf_counter()

    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        loss = model(**batch).loss

        if torch.isnan(loss):
            print(f'NaN loss at epoch {epoch}, step {steps} — skipping batch')
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        steps      += 1

    avg_loss = total_loss / max(steps, 1)
    acc, macro_f1, _, _ = evaluate(model, val_loader)
    elapsed = time.perf_counter() - epoch_start

    target_hit = acc >= 0.88 and macro_f1 >= 0.85
    status     = '*** TARGET HIT ***' if target_hit else ''
    print(f"{epoch:<7} {avg_loss:<13.4f} {acc*100:<12.2f} {macro_f1:<12.4f} {status}  ({elapsed:.0f}s)")

    # Save best
    if macro_f1 > best_f1:
        best_f1, best_epoch, no_improve = macro_f1, epoch, 0
        model.save_pretrained('./deberta-intent-spotify-best')
        tokenizer.save_pretrained('./deberta-intent-spotify-best')
        print(f'       ↑ New best F1={best_f1:.4f} — model saved')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'\nEarly stop: no F1 improvement for {PATIENCE} epochs.')
            break

total_time = time.perf_counter() - train_start
print(f'\n[DONE] Best Macro F1: {best_f1:.4f} at epoch {best_epoch} | Total time: {total_time/60:.1f} min')

### Cell 6 — Gold Benchmark (200 curated test cases)

In [ ]:
# Reload best model from disk
model = AutoModelForSequenceClassification.from_pretrained(
    './deberta-intent-spotify-best').to(DEVICE)

gold_records = [json.loads(l) for l in open('golden_eval.jsonl', encoding='utf-8')]
gold_texts   = [r.get('message') or r.get('customer_message') for r in gold_records]
gold_labels  = [INTENT_MAP[r['intent']] for r in gold_records]

model.eval()
predictions = []
start = time.perf_counter()

with torch.no_grad():
    for i in range(0, len(gold_texts), 32):
        batch_texts = gold_texts[i:i+32]
        enc = tokenizer(
            batch_texts, return_tensors='pt', truncation=True,
            max_length=MAX_LEN, padding=True
        )
        # Send only input_ids + attention_mask (no token_type_ids)
        inputs = {
            'input_ids':      enc['input_ids'].to(DEVICE),
            'attention_mask': enc['attention_mask'].to(DEVICE)
        }
        out = model(**inputs)
        predictions.extend(torch.argmax(out.logits, dim=-1).cpu().tolist())

elapsed  = time.perf_counter() - start
acc      = accuracy_score(gold_labels, predictions)
macro_f1 = f1_score(gold_labels, predictions, average='macro', zero_division=0)

print('=' * 68)
print('   DEBERTA INTENT CLASSIFIER — GOLD BENCHMARK RESULTS')
print('=' * 68)
print(f'  Baseline Accuracy  (rule-based):   51.50%')
print(f'  Fine-Tuned Accuracy:               {acc*100:.2f}%   {"[TARGET HIT]" if acc>=0.88 else "[need >= 88%]"}')
print(f'  Macro F1 Score:                    {macro_f1:.4f}  {"[TARGET HIT]" if macro_f1>=0.85 else "[need >= 0.85]"}')
print(f'  Latency per query:                 {(elapsed/len(gold_texts))*1000:.2f} ms')
print(f'  Total gold examples:               {len(gold_records)}')
print('=' * 68)
print('\nPer-Class Breakdown:')
target_names = [ID_TO_LABEL[i] for i in range(NUM_LABELS)]
print(classification_report(gold_labels, predictions, target_names=target_names, digits=4, zero_division=0))

### Cell 7 — Download Model to Your PC

In [ ]:
from google.colab import files
print('Zipping model...')
!zip -r deberta-intent-spotify-best.zip deberta-intent-spotify-best/
size_mb = os.path.getsize('deberta-intent-spotify-best.zip') / 1e6
print(f'Archive: {size_mb:.1f} MB — starting download...')
files.download('deberta-intent-spotify-best.zip')
print('\nOnce downloaded, unzip to: E:/Reply_agent/models/deberta-intent-spotify-best/')